In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7807543994986015, 'n_it': 0.37524585933731774}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.708799669322426, 14.755143077917666, 15.460970748365678, 14.078027353246915, 14.919163710690466, 14.605153377723243, 13.881494951101043, 13.905591565761688, 16.430254305953667, 14.699432708931173, 15.753734455667109, 16.49823569685272, 15.01466612871424, 16.903777066736247, 16.95402653080736, 15.035133502424495, 15.339830082472089, 15.377006710235442, 13.593925683321542, 13.93298772363275, 14.218047293357927, 13.864316905437017, 14.420744842611482, 14.70791130877276, 13.635949717009353, 13.624754901425899, 16.636787748135873, 18.096800460231798, 17.37360524714137, 13.888034049067185, 13.79087088272572, 16.551719719834146, 14.785672030717604, 17.233749181643024, 14.668884795913597, 14.218748521433998, 14.462377617107775, 14.177596989195113, 14.013481634608281, 12.939678484911514, 14.000416998673414, 14.018044569229968, 17.65977810819307, 16.363180888831906, 13.613720064691543, 13.90665711202007, 15.653252586564218, 15.019404102095233, 17.682818299748206, 13.62651122056203, 13.947549

In [5]:
np.average(y_max_arr)

np.float64(14.784896423104406)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M05/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)